In [ ]:
#@title Copyright 2023 Google LLC. Double-click for license information.
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Colabs

Machine Learning Crash Course uses Colaboratories (Colabs) for all programming exercises. Colab is Google's implementation of [Jupyter Notebook](https://jupyter.org/). For more information about Colabs and how to use them, go to [Welcome to Colaboratory](https://research.google.com/colaboratory).

# Numerical data: Statistics on a dataset

This Colab programming exercise (first of two) is part of the Machine Learning Crash Course module [Working with numerical data](https://developers.google.com/machine-learning/crash-course/numerical-data).

## What to expect

In the section, [First steps with numerical data](https://developers.google.com/machine-learning/crash-course/numerical-data/first-steps), you learned how to do the following:
- Visualize your data in plots or graphs.
- Evaluate potential features and labels mathematically.
- Find [**outliers**](https://developers.google.com/machine-learning/glossary/#outliers) in the dataset.

This exercise takes you through the process of finding columns that contain blatant outliers, which you can then decide to keep in or delete from the dataset.

In [ ]:
# @title Setup - Install relevant modules

!pip install pandas~=2.2.0

In [ ]:
# @title Setup - Import relevant modules

# The following code imports relevant modules that
# allow you to run the colab.
# If you encounter technical issues running some of the code sections
# that follow, try running this section again.

import pandas as pd

# The following lines adjust the granularity of reporting.
pd.options.display.max_rows = 10
pd.options.display.float_format = "{:.1f}".format

In [ ]:
#@title Import the dataset

# The following code imports the dataset that is used in the colab.

df = pd.read_csv(filepath_or_buffer="https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv")

## Get basic statistics

In the following code section, the DataFrame `describe` method returns basic statistics on all the columns in the dataset, such as:

* `count` is the number of populated elements in this column. Ideally, every column contains the same value for `count`, but that's not always the case.
* `mean` is the traditional average of values in that column. We recommend comparing the `mean` to the median for each column. The **median** is the 50% row of the table.
* `std` is the standard deviation of the values in this column.
* `min`, `25%`, `50%`, `75%`, and `max` indicate values in the 0, 25, 50, 75, and 100th percentiles.

In [ ]:
# Get statistics on the dataset.

# The following code returns basic statistics about the data in the dataframe.

df.describe()

### Task: Identify possible outliers

Based on the preceding statisics, do you see any columns that might contain outliers?

In [ ]:
# @title Solution (run this code block to view) { display-mode: "form" }

print("""The following columns might contain outliers:

  * total_rooms
  * total_bedrooms
  * population
  * households
  * possibly, median_income

In all of those columns:

  * the standard deviation is almost as high as the mean
  * the delta between 75% and max is much higher than the
      delta between min and 25%.""")

# **Problem Overview and Dataset Description**

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.columns

#Feature Engineering: Creating Housing Density and Room Ratio Features:

In [ ]:
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_household'] = df['total_bedrooms'] /df['total_rooms']
df['people_per_house'] = df['population'] / df['households']

#Feature Selection: Removing Raw Features

In [ ]:
df.drop(['total_rooms', 'total_bedrooms', 'population'], axis=1, inplace=True)

In [ ]:
df

In [ ]:
df.describe()

#Feature Engineering: Distance and Regional Features

In [ ]:
df['distance_to_coast'] = abs(df['longitude'] + 122)
df['dist_to_sf'] = ((df['longitude'] + 122.42)**2 + (df['latitude'] - 37.77)**2) ** 0.5
df['dist_to_ls'] = ((df['longitude'] + 118.24)**2+(df['latitude'] - 34.05)**2)**0.5
df['north_california'] = (df['latitude'] > 36).astype(int)

#Feature Selection: Dropping Geographic Coordinates

In [ ]:
df.drop(['longitude', 'latitude'], axis=1, inplace=True)

In [ ]:
df

#Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df.hist(figsize=(12, 10))
plt.show()

In [ ]:
sns.boxplot(x=df['rooms_per_household'])
plt.show()

In [ ]:
df[df['rooms_per_household'] > 50]

In [ ]:
import numpy as np

In [ ]:
df['rooms_per_household'] = np.log1p(df['rooms_per_household'])

In [ ]:
sns.boxplot(x=df['people_per_house'])

In [ ]:
df[df['people_per_house'] > 20]

In [ ]:
df['people_per_house'] = df['people_per_house'].clip(upper=20)

In [ ]:
sns.boxplot(x=df['median_income'])

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
plt.figure(figsize=(12, 10), dpi=500)
sns.scatterplot(x=df['median_income'], y=df['median_house_value'])
plt.show()

In [ ]:
import plotly.express as px

In [ ]:
px.scatter_3d(df, x=df['median_income'], y=df['rooms_per_household'], z=df['median_house_value']).show()

#Train-Test Split

In [ ]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=41, test_size=0.2)

##Linear Regression: Model Training and Evaluation

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
scaler = StandardScaler()

In [ ]:
model = LinearRegression()

In [ ]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

In [ ]:
rmse

In [ ]:
coeffs = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_
})

coeffs.sort_values("coefficient", ascending=False)

#Random Forest: Model Training and Evaluation

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model_rf = RandomForestRegressor(random_state=41)

In [ ]:
model_rf.fit(X_train, y_train)

In [ ]:
pred = model_rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))

In [ ]:
rmse

In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model_rf.feature_importances_
}).sort_values("importance", ascending=False)

In [ ]:
print(importance)

#XGBoost Regressor: Model Training and Evaluation

In [ ]:
from xgboost import XGBRegressor

In [ ]:
model_xgb = XGBRegressor(n_estimators=300, random_state=41, learning_rate=0.05, max_depth=5)

In [ ]:
model_xgb.fit(X_train, y_train)

In [ ]:
pred_y = model_xgb.predict(X_test)

rmse = np.sqrt(mean_squared_error(pred_y, y_test))

In [ ]:
rmse

#Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
model = XGBRegressor(random_state=42)

In [ ]:
params = {
    'n_estimators': [200, 300, 400],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1],
    'colsample_bytree': [0.7, 0.8, 0.9, 1]
}

In [ ]:
final_model = RandomizedSearchCV(model, params, n_iter=20, cv=3, scoring='neg_root_mean_squared_error', verbose=2)

In [ ]:
final_model.fit(X_train, y_train)

#Final Model Performance and Feature Importance

In [ ]:
y_pred = final_model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

In [ ]:
rmse

In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": final_model.best_estimator_.feature_importances_
}).sort_values('importance', ascending=False)

In [ ]:
sns.barplot(
    x="importance",
    y="feature",
    data=importance
)

plt.title("XGBoost Feature Importance")
plt.show()

The tuned XGBoost model achieved an RMSE of 45,265 on the validation set.
Feature importance analysis shows that median_income and geographic features
are the strongest predictors of housing prices.

